# 06 · Update the model card on Hugging Face

Notebook 04 writes a model card (`README.md`) for the uploaded model. The first version of that template had two mistakes:
1. It always said "class-weighted cross-entropy", even when the selected run was the *without class weights* ablation.
2. It linked to a Hugging Face Space for the demo. Gradio Spaces now need a paid PRO account, so the app is deployed on Streamlit Community Cloud instead, and its link is in the GitHub README.

This notebook corrects both. It needs no GPU, takes under a minute, and asks for your Hugging Face **write** token (https://huggingface.co/settings/tokens).

In [ ]:
# --- Setup: works both locally and in Google Colab ---
import os, sys
REPO_URL = "https://github.com/Joellate/kiswahili-sentiment-analysis.git"

if "google.colab" in sys.modules:
    if not os.path.exists("kiswahili-sentiment-analysis"):
        !git clone -q $REPO_URL
    %cd kiswahili-sentiment-analysis
    %pip install -q -r requirements.txt
else:
    # running locally from notebooks/
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, "src")
print("Working directory:", os.getcwd())

In [ ]:
import re
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download, login, whoami

login(token=getpass("Paste your Hugging Face WRITE token and press Enter: ").strip())
MODEL_REPO = f"{whoami()['name']}/kiswahili-sentiment"
api = HfApi()

card = open(hf_hub_download(MODEL_REPO, "README.md", force_download=True), encoding="utf-8").read()
new = card
if "without class weights" in new:
    new = new.replace("class-weighted cross-entropy", "unweighted cross-entropy (no class weights)")
new = re.sub(r"Demo: https://huggingface\.co/spaces/\S+",
             "Live demo: see the link at the top of https://github.com/Joellate/kiswahili-sentiment-analysis", new)

if new != card:
    api.upload_file(path_or_fileobj=new.encode("utf-8"), path_in_repo="README.md", repo_id=MODEL_REPO,
                    commit_message="Model card: correct loss description and demo link")
    print("Model card updated: https://huggingface.co/" + MODEL_REPO)
else:
    print("Model card already up to date")
print(new)